In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import osmnx as ox

# ----------------------------
# 1. LOAD DATA
# ----------------------------

# Census sections (sezioni di censimento) R03 = Lombardia, R09 = Toscana
sections = gpd.read_file("R03_11_WGS84/R03_11_WGS84.shp") # Call only the shp, but the shp, shx, dbf, and prj have to be together wherever called from.
# Sistema Locale di Lavoro (SLL) mapping (municipality -> SLL)
sll = pd.read_excel("RACCORDO COMUNI2011_SLL2011.xlsx", sheet_name="COM_SLL_2011")

# Origin-destination matrix 
# column specs: (start, end) in Python indexing (0-based, end-exclusive)
colspecs = [
    (0, 1),    # record type S or L
    (2, 3),    # residence type
    (4, 7),    # province of residence
    (8, 11),   # municipality of residence
    (13, 14),  # assigned sex of respondent
    (15, 16),  # reason for travel, 1 for education, 2 for work
    (17, 18),  # workplace location type
    (19, 22),  # province of work or education
    (23, 26),  # municipality of work or education
    (50, 60),  # number of individuals
]

columns = [
    "record_type",
    "residence_type",
    "prov_res",
    "com_res",
    "sex",
    "reason",
    "work_loc_type",
    "prov_work",
    "com_work",
    "n_individuals" #living in area A and working in area A
]

odmatrix = pd.read_fwf(
    "matrix_pendo2011_10112014.txt",
    colspecs=colspecs,
    names=columns,
    dtype=str  # keep raw first
)

# Keep only "S" records
odmatrix = odmatrix[odmatrix["record_type"] == "S"]

"""# Keep only work trips
odmatrix = odmatrix[odmatrix["reason"] == "2"]"""

# Convert counts
odmatrix["n_individuals"] = pd.to_numeric(odmatrix["n_individuals"], errors="coerce")

odmatrix.head()

ModuleNotFoundError: No module named 'osmnx'

In [ ]:
# ----------------------------
# 1. Convert PRO_COM in shp from float into *6-digit* ISTAT code as string for join
# ----------------------------

sections["PRO_COM"] = (
    sections["PRO_COM"]
    .astype(int)        # remove .0
    .astype(str)        # back to string
    .str.zfill(6)       # restore leading zeros
)

sections.head()
print(sections.columns[:10])
print(sections.shape)
print(sections["PRO_COM"].head())
print(sections["COD_ISTAT"].head()) #Not what we want here, the region code then the 6-digit ISTAT code

In [ ]:
# Compute ISTAT codes for the origin-destination matrix
odmatrix["prov_work"] = odmatrix["prov_work"].astype(str)
odmatrix["prov_res"] = odmatrix["prov_res"].astype(str)
odmatrix["com_work"] = odmatrix["com_work"].astype(str)
odmatrix["com_res"] = odmatrix["com_res"].astype(str)
odmatrix['ISTATcode_res']=odmatrix['prov_res'].str.zfill(3) + odmatrix['com_res'].str.zfill(3)
odmatrix['ISTATcode_work']=odmatrix['prov_work'].str.zfill(3) + odmatrix['com_work'].str.zfill(3)
odmatrix.head()

In [ ]:
# ----------------------------
# 2. Clean SLL key file and create 6-digit ISTAT ID for join
# ----------------------------
desired_sll = str(313)
# Convert municipality codes to strings
sll["SLL_2011"] = sll["SLL_2011"].astype(int).astype(str)
sll["ISTAT_2011"] = sll["ISTAT_2011"].astype(int).astype(str)
sll["ISTAT_2011_full"] = sll["ISTAT_2011"].astype(int).astype(str).str.zfill(6)
sll = sll.rename(columns={"POP 2011": "POP_2011"})
sll = sll[sll["SLL_2011"] == desired_sll]
sll.head()

In [ ]:
# ----------------------------
# 3. Filter municipalities by ISTAT code in SLL file to only those in desired SLL
# ----------------------------

# Select Milan SLL
sll_milan = sll[sll["SLL_2011"] == "313"]

milan_municipalities = sll_milan["ISTAT_2011_full"]

# Filter sections
filtered_sections = sections[sections["PRO_COM"].isin(milan_municipalities)].copy()

# Filter OD matrix
filtered_odmatrix = odmatrix[odmatrix["ISTATcode_work"].isin(milan_municipalities)].copy()

In [ ]:
filtered_sections.head()

In [ ]:
# ----------------------------
# 4. COMPUTE JOBS BY MUNICIPALITY (WORKPLACE-BASED)
# ----------------------------
jobs_by_muni = (
    filtered_odmatrix.groupby("ISTATcode_work")["n_individuals"]
    .sum()
    .reset_index()
    .rename(columns={"n_individuals": "jobs_municipality"})

)

In [ ]:
# ----------------------------
# 5. MERGE JOBS INTO SECTIONS
# ----------------------------
filtered_sections = filtered_sections.merge(
    jobs_by_muni,
    left_on="PRO_COM",
    right_on="ISTATcode_work",
    how="left"
)

filtered_sections["n_individuals"] = filtered_sections["n_individuals"].fillna(0)
filtered_sections.head()

In [ ]:
# ----------------------------
# 6. CREATE WEIGHTS (POPULATION-BASED)
# ----------------------------

# Total population per municipality
filtered_sections["municipality_population"] = (
    filtered_sections.groupby("PRO_COM")["n_individuals"]
    .transform("sum")
)

# Weight of each section within its municipality
filtered_sections["weight"] = (
    filtered_sections["n_individuals"] / filtered_sections["municipality_population"]
)

In [ ]:
# ----------------------------
# 7. ALLOCATE JOBS TO SECTIONS
# ----------------------------

filtered_sections["estimated_jobs"] = (
    filtered_sections["jobs_municipality"] * filtered_sections["weight"]
)

In [ ]:
# ----------------------------
# 8. COMPUTE AREA AND DENSITY
# ----------------------------

# Reproject to a metric CRS (Europe-wide projection)
filtered_sections = filtered_sections.to_crs(epsg=3035)

# Area in km²
filtered_sections["area_km2"] = filtered_sections.geometry.area / 1_000_000

# Employment density (jobs per km²)
filtered_sections["employment_density"] = (
    filtered_sections["estimated_jobs"] / filtered_sections["area_km2"]
)

In [ ]:
# Define place
place = "Milan, Italy"

# Get rail network (includes metro, tram, rail)
tags = {
    "railway": [
        "rail",        # main rail
        "subway",      # metro
        "light_rail",  # S lines often tagged here
        "tram"
    ]
}

gdf = ox.features_from_place(place, tags)

# Keep only lines (not stations)
gdf = gdf[gdf.geometry.type.isin(["LineString", "MultiLineString"])]
gdf = gdf.to_crs(sections.crs)
gdf.head()

In [ ]:
# ----------------------------
# 10. QUICK VISUALIZATION
# ----------------------------

fig, ax = plt.subplots(figsize=(10, 10))

filtered_sections.plot(
    column="employment_density",
    scheme="quantiles",
    k=5,
    legend=True,
    ax=ax
)

ax.set_title("Estimated Employment Density - Milan SLL")
ax.axis("off")

plt.show()